# Customer Behaviour Purchase

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [2]:
dt=pd.read_csv("shop_smart_ecommerce.csv")
dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [3]:
le=LabelEncoder()
dt["Month"]=le.fit_transform(dt["Month"])
dt["VisitorType"]=le.fit_transform(dt["VisitorType"])
dt["Weekend"]=le.fit_transform(dt["Weekend"])
dt["Revenue"]=le.fit_transform(dt["Revenue"])
dt

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.000000,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.000000,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.000000,0.0,2,3,3,1,4,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,0.007143,0.029031,12.241717,0.0,1,4,6,1,1,2,1,0
12326,0,0.0,0,0.0,5,465.750000,0.000000,0.021333,0.000000,0.0,7,3,2,1,8,2,1,0
12327,0,0.0,0,0.0,6,184.250000,0.083333,0.086667,0.000000,0.0,7,3,2,1,13,2,1,0
12328,4,75.0,0,0.0,15,346.000000,0.000000,0.021053,0.000000,0.0,7,2,2,3,11,2,0,0


In [4]:
X=dt.drop(columns=["Revenue"],axis=1)
y=dt["Revenue"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,accuracy_score
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train,y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [6]:
y_pred=model.predict(X_test)

print("F1 Score:",f1_score(y_test,y_pred))
print("accuracy_score:",accuracy_score(y_test,y_pred))

F1 Score: 0.5302642796248934
accuracy_score: 0.8510408218437415


In [7]:
# Plot Tree
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [8]:
path=full_tree.cost_complexity_pruning_path(X_train,y_train)
ccp_alphas=path.ccp_alphas


In [9]:
# Now I have to find best model
trees=[]

for alpha in ccp_alphas:
    model=DecisionTreeClassifier(random_state=42,ccp_alpha=alpha)
    model.fit(X_train,y_train)

    trees.append((model,alpha))



In [10]:
best_acc=0
best_alpha=0
for model,alpha in trees:
    curr_acc=model.score(X_test,y_test)
    if(best_acc<curr_acc):
        best_acc=curr_acc
        best_alpha=alpha
    
    
    

In [12]:
print(best_acc)
print(best_alpha*100)


0.8961881589618816
0.06486171110780081


In [13]:
best_model = DecisionTreeClassifier(random_state=42,ccp_alpha=best_alpha)
best_model.fit(X_train,y_train)
y_pred=best_model.predict(X_test)

print("F1 Score:",f1_score(y_test,y_pred))
print("accuracy_score:",accuracy_score(y_test,y_pred))

F1 Score: 0.6175298804780877
accuracy_score: 0.8961881589618816


# With Industry Best practices solving this problem 

In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import f1_score,classification_report,confusion_matrix

In [34]:
df=pd.read_csv("shop_smart_ecommerce.csv")
X=df.drop(columns="Revenue",axis=1)
y=df["Revenue"].astype(int) # Because True/false already having 1 & 0 value so its a waste of time and speed converting by label encoder or hot encode


In [36]:
# From here we find the data type of the features which help us to standardized sale and apply labelEncode and hotEncoding
num_features=X.select_dtypes(include=["int64","float64"]).columns
cat_features=X.select_dtypes(include=["object","category"]).columns


In [37]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #unbalance data hai to jb test krenge tb problem ho sakta hai islye ham ratio maintain rakte hai 
    )


In [46]:
# Preprocessin Pipeline
# Yha kya ho rha different dtypes pe different transformation lga hai jaise hotEncoding ya scaling 
preprocessor=ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),num_features),
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat_features)
            ]
)

In [50]:
dt=DecisionTreeClassifier(
    max_depth=6,            # Prevent deep Overfitting
    min_samples_leaf=30,      # smooth decision boundaries
    class_weight="balanced", # handles imbalance
    random_state=42
)
# class_weight="balanced" automatically adjusts weights inversely proportional to class frequencies so that minority classes receive higher importance during model training.

In [53]:
pipe=Pipeline(
    steps=[
        ("preprocess",preprocessor),
        ("model",dt)
    ]
)

In [54]:
pipe.fit(X_train,y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [56]:
y_pred=pipe.predict(X_test)
print("F1 score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

F1 score: 0.6278381046396841

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.85      0.90      2084
           1       0.50      0.83      0.63       382

    accuracy                           0.85      2466
   macro avg       0.73      0.84      0.77      2466
weighted avg       0.89      0.85      0.86      2466


Confusion Matrix:
 [[1771  313]
 [  64  318]]


# HyperParameter Tuning 

In [ ]:
# Machine Learning model ko train karte waqt kuch settings hoti hain jo model ka behaviour control karti hain.
# In settings ko bolte hain Hyperparameters.

# Example (Decision Tree):

# max_depth
# min_samples_split
# min_samples_leaf
# ccp_alpha

# Inka best value find karna hi Hyperparameter Tuning kehlata hai.

In [61]:
from sklearn.model_selection import GridSearchCV

param_grid={
    "model__max_depth":[4,6,8],
    "model__min_samples_leaf":[20,30,50]
}
grid=GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)
grid.fit(X_train,y_train)
print("Best F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Best F1: 0.6343735129725652
Best params: {'model__max_depth': 4, 'model__min_samples_leaf': 50}


In [ ]:
# GridSearchCV performs hyperparameter tuning by testing multiple parameter 
# combinations using cross-validation and 
# selecting the combination that gives the best evaluation score.